In [ ]:
from datetime import datetime

job_id = session.sql(
    "SELECT UUID_STRING()"
).collect()[0][0]

job_name = "DWH_FACT_MEDICATIONS_LOAD"
layer_name = "DWH"

start_time = datetime.now()

rows_loaded = 0
rows_failed = 0

run_status = "SUCCESS"
error_message = None

try:

    fact_medications_df = session.sql(f"""

        SELECT

            m.MED_ID,
            d.MEDICATIONS_INFO_ID,
            m.COMMON_NAME,
            m.ROGER_DECISION_DRUG_IND,
            m.MEDICATION_DESC,
            m.ALLOW_FREQUENCY_NUM,
            m.ADD_PERSON_ID,
            m.ADD_ORGN_ID,
            m.MOD_PERSON_ID,
            m.MOD_ORGN_ID,
            m.ADD_TS,
            m.ADD_USER,
            m.MOD_TS,
            m.MOD_USER,
            CURRENT_TIMESTAMP() AS CREATED_DATE,
            m.DELETED_DATE

        FROM {STG}.{STG_MEDICATIONS} m

        INNER JOIN {DWH}.{DIM_MEDICATIONS_INFO} d
            ON m.MEDICATION_CLASSIFICATION_CODE = d.MEDICATION_CLASSIFICATION_CODE_AV_ID
           AND d.UPDATED_DATE IS NULL

    """)

    fact_medications_df.create_or_replace_temp_view(
        "TEMP_FACT_MEDICATIONS"
    )

    merge_result = session.sql(f"""

        MERGE INTO {DWH}.{FACT_MEDICATIONS} tgt
        USING TEMP_FACT_MEDICATIONS src
            ON tgt.MED_ID = src.MED_ID
        WHEN MATCHED AND (
            src.MOD_TS > tgt.MOD_TS
            OR (tgt.MOD_TS IS NULL AND src.MOD_TS IS NOT NULL)
            OR src.ADD_TS > tgt.ADD_TS
        ) THEN
            UPDATE SET
                MEDICATIONS_INFO_ID = src.MEDICATIONS_INFO_ID,
                COMMON_NAME = src.COMMON_NAME,
                ROGER_DECISION_DRUG_IND = src.ROGER_DECISION_DRUG_IND,
                MEDICATION_DESC = src.MEDICATION_DESC,
                ALLOW_FREQUENCY_NUM = src.ALLOW_FREQUENCY_NUM,
                ADD_PERSON_ID = src.ADD_PERSON_ID,
                ADD_ORGN_ID = src.ADD_ORGN_ID,
                MOD_PERSON_ID = src.MOD_PERSON_ID,
                MOD_ORGN_ID = src.MOD_ORGN_ID,
                ADD_TS = src.ADD_TS,
                ADD_USER = src.ADD_USER,
                MOD_TS = src.MOD_TS,
                MOD_USER = src.MOD_USER,
                DELETED_DATE = src.DELETED_DATE
        WHEN NOT MATCHED THEN
            INSERT (
                MED_ID,
                MEDICATIONS_INFO_ID,
                COMMON_NAME,
                ROGER_DECISION_DRUG_IND,
                MEDICATION_DESC,
                ALLOW_FREQUENCY_NUM,
                ADD_PERSON_ID,
                ADD_ORGN_ID,
                MOD_PERSON_ID,
                MOD_ORGN_ID,
                ADD_TS,
                ADD_USER,
                MOD_TS,
                MOD_USER,
                CREATED_DATE,
                DELETED_DATE
            )
            VALUES (
                src.MED_ID,
                src.MEDICATIONS_INFO_ID,
                src.COMMON_NAME,
                src.ROGER_DECISION_DRUG_IND,
                src.MEDICATION_DESC,
                src.ALLOW_FREQUENCY_NUM,
                src.ADD_PERSON_ID,
                src.ADD_ORGN_ID,
                src.MOD_PERSON_ID,
                src.MOD_ORGN_ID,
                src.ADD_TS,
                src.ADD_USER,
                src.MOD_TS,
                src.MOD_USER,
                src.CREATED_DATE,
                src.DELETED_DATE
            )

    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_MEDICATIONS load completed. Rows Inserted: {rows_inserted}, Rows Updated: {rows_updated}"
    )

    print(
        f"DWH LOAD SUCCESS | Rows Inserted: {rows_inserted} | Rows Updated: {rows_updated}"
    )

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1

    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_MEDICATIONS load failed. Error: {error_message}"
    )

    print(
        f"DWH LOAD FAILED: {error_message}"
    )